In [ ]:
# ============================================================
# Hybrid Rocket Fuel Blend Analysis  —  LOO by firing
# Inputs  : X_Value, Force, Pressure_0, Gox
# Outputs : Pressure, rdot_mm_s, Isp, Thrust, cstar
# Model   : XGBoost multi-output + MC Dropout (per LOO fold)
# ============================================================
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [ ]:
# ============================================================
# CONFIG
# ============================================================
EXCEL_PATH = Path("daq_raw_4blends.xlsx")
OUTPUT_EXCEL = Path("Best_Fuel_Results_xgboost_LOO.xlsx")
OUTPUT_TXT = Path("Best_Fuel_Summary_xgboost_LOO.txt")
OUTPUT_GRAPH = Path("Gox_vs_Predicted_Regression_Rate_xgboost_LOO.png")
OUTPUT_GRAPH1 = Path("Pressure_vs_Predicted_Regression_Rate_xgboost_LOO.png")

SHEETS_TO_USE = None
REQUIRED_SHEET_COUNT = 4
FUEL_COLUMN = "fuel_type"
MODEL_NAME = "XGBoost"

FEATURE_COLUMNS = ["X_Value", "Force", "Pressure_0", "Gox"]
TARGET_COLUMNS = ["Pressure", "rdot_mm_s", "Isp", "Thrust", "cstar"]
TARGET_WEIGHTS = {t: 1.0 for t in TARGET_COLUMNS}

RDOT_TARGET = "rdot_mm_s"
PRED_RDOT = f"pred_{RDOT_TARGET}"

SEED = 100
VALIDATION_SIZE = 0.20
N_MC_SAMPLES = 100
UNCERTAINTY_PENALTY_K = 0.5
MIN_LABELED_ROWS = 4

N_ESTIMATORS = 300
MAX_DEPTH = 3
LEARNING_RATE = 0.05
REG_ALPHA = 1.0
REG_LAMBDA = 5.0
SUBSAMPLE = 0.8
COLSAMPLE = 0.8
MIN_CHILD_WEIGHT = 5

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ============================================================
# 2. HELPERS
# ============================================================

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def clean_numeric(df, columns):
    out = df.copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan
    return out


def safe_r2_multi(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        if np.unique(y_true[:, i]).size < 2:
            scores.append(np.nan)
        else:
            scores.append(r2_score(y_true[:, i], y_pred[:, i]))
    return np.asarray(scores, dtype=float)


def minmax_norm(series):
    low, high = series.min(), series.max()
    if not np.isfinite(low) or not np.isfinite(high) or abs(high - low) < 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - low) / (high - low)


def mc_dropout_predict(model, x_input, n_samples):
    preds = np.empty((n_samples, len(x_input)), dtype=np.float32)
    for i in range(n_samples):
        preds[i] = model(x_input, training=True).numpy().ravel()
    return preds.mean(axis=0), preds.std(axis=0)


def validate_excel_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Excel not found: {path.resolve()}\n"
            "Put your data Excel next to this script or set EXCEL_PATH."
        )
    if path.stat().st_size < 100:
        raise ValueError(f"File is too small: {path}")
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:2] != b"PK":
        raise ValueError(
            f"Not a valid .xlsx: {path.resolve()}\n"
            "Save As → Excel Workbook (.xlsx) and retry."
        )


In [ ]:
# ============================================================
# 3. LOAD  (frames are built inside the loop)
# ============================================================

def load_dataset():
    validate_excel_file(EXCEL_PATH)
    try:
        xls = pd.ExcelFile(EXCEL_PATH, engine="openpyxl")
    except Exception as exc:
        raise ValueError(f"Could not open Excel: {EXCEL_PATH.resolve()}\n{exc}") from exc

    sheet_names = xls.sheet_names if SHEETS_TO_USE is None else list(SHEETS_TO_USE)
    if len(sheet_names) < REQUIRED_SHEET_COUNT:
        raise ValueError(
            f"Need >= {REQUIRED_SHEET_COUNT} sheets; found {len(sheet_names)}. "
            f"Sheets: {xls.sheet_names}"
        )
    sheet_names = sheet_names[:REQUIRED_SHEET_COUNT]
    frames = []
    gox_col_global = "Gox" if "Gox" in FEATURE_COLUMNS else None

    for sheet in sheet_names:
        raw = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")
        missing_in = [c for c in FEATURE_COLUMNS if c not in raw.columns]
        if missing_in:
            raise ValueError(
                f"Sheet '{sheet}' missing INPUT columns: {missing_in}\n"
                f"Columns present: {list(raw.columns)}"
            )
        keep = list(FEATURE_COLUMNS)
        for t in TARGET_COLUMNS:
            if t not in raw.columns:
                raw[t] = np.nan
            if t not in keep:
                keep.append(t)
        keep_unique = list(dict.fromkeys(keep))
        frame = clean_numeric(raw[keep_unique], keep_unique)
        frame[FUEL_COLUMN] = str(sheet)
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)
    data = data.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
    if "X_Value" in data.columns:
        data = data[data["X_Value"] >= 0].reset_index(drop=True)
    if data.empty:
        raise ValueError("No rows with valid INPUT columns after cleaning.")

    labeled_mask = data[TARGET_COLUMNS].notna().all(axis=1)
    for t in TARGET_COLUMNS:
        labeled_mask &= np.isfinite(pd.to_numeric(data[t], errors="coerce"))

    n_labeled = int(labeled_mask.sum())
    print(f"Sheets used      : {sheet_names}")
    print(f"Total input rows : {len(data)}")
    print(f"Labeled rows     : {n_labeled}")
    print(f"Unlabeled rows   : {len(data) - n_labeled}")
    print(f"Inputs           : {FEATURE_COLUMNS}")
    print(f"Targets          : {TARGET_COLUMNS}")
    print(f"Validation       : Leave-One-Firing-Out (LOO by {FUEL_COLUMN})")
    print(f"Main model       : {MODEL_NAME}")

    if n_labeled < MIN_LABELED_ROWS:
        raise ValueError(
            f"Need at least {MIN_LABELED_ROWS} labeled rows; found {n_labeled}.\n"
            f"Fill targets {TARGET_COLUMNS} or lower MIN_LABELED_ROWS."
        )
    n_groups = data.loc[labeled_mask, FUEL_COLUMN].nunique()
    if n_groups < 2:
        raise ValueError("LOO by firing needs at least 2 different fuel_type / sheets in labeled data.")
    return data, sheet_names, labeled_mask, gox_col_global


In [ ]:
# ============================================================
# 4. MC DROPOUT
# ============================================================

def build_mc_dropout_inputs(x_train, x_other, feature_cols):
    encoder = make_one_hot_encoder()
    train_fuel = encoder.fit_transform(x_train[[FUEL_COLUMN]])
    other_fuel = encoder.transform(x_other[[FUEL_COLUMN]])
    scaler = StandardScaler()
    train_num = scaler.fit_transform(x_train[feature_cols]).astype(np.float32)
    other_num = scaler.transform(x_other[feature_cols]).astype(np.float32)
    return (
        np.hstack([train_num, train_fuel]).astype(np.float32),
        np.hstack([other_num, other_fuel]).astype(np.float32),
        encoder,
        scaler,
    )


def train_mc_dropout_network(x_train_nn, y_train_rdot):
    # small val split only for EarlyStopping (from train fold)
    if len(x_train_nn) < 5:
        x_fit, y_fit = x_train_nn, y_train_rdot
        x_val, y_val = x_train_nn, y_train_rdot
    else:
        x_fit, x_val, y_fit, y_val = train_test_split(
            x_train_nn, y_train_rdot, test_size=VALIDATION_SIZE, random_state=SEED
        )
    model = Sequential([
        Input(shape=(x_train_nn.shape[1],)),
        Dense(64, activation="relu"),
        Dropout(0.30),
        Dense(32, activation="relu"),
        Dropout(0.30),
        Dense(16, activation="relu"),
        Dropout(0.20),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.002), loss="mse")
    es = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0)
    model.fit(
        x_fit, y_fit,
        validation_data=(x_val, y_val),
        epochs=200,
        batch_size=min(64, max(8, len(x_fit))),
        callbacks=[es],
        verbose=0,
    )
    return model


In [ ]:
def print_metrics_table(test_r2, test_mae, test_rmse, targets, title="LOO METRICS"):
    """Evenly spaced metrics table (fixed column widths)."""
    # header
    header = f"{'Target':<14}{'R2':>12}{'MAE':>12}{'RMSE':>12}"
    sep = "-" * len(header)
    print(f"\n========== {title} ==========")
    print(header)
    print(sep)
    rows = []
    for i, t in enumerate(targets):
        r2 = test_r2[i]
        r2_s = f"{r2:.4f}" if np.isfinite(r2) else "nan"
        mae_s = f"{test_mae[i]:.4f}"
        rmse_s = f"{test_rmse[i]:.4f}"
        print(f"{t:<14}{r2_s:>12}{mae_s:>12}{rmse_s:>12}")
        rows.append({"Target": t, "R2": r2_s, "MAE": mae_s, "RMSE": rmse_s})
    print(sep)
    return pd.DataFrame(rows)


def print_ranking_table(summary, pred_columns, title="FUEL RANKING (LOO)"):
    """Evenly spaced ranking table (fixed column widths)."""
    cols = ["rank", FUEL_COLUMN] + list(pred_columns) + ["uncertainty_rdot", "final_score"]
    cols = [c for c in cols if c in summary.columns]
    view = summary[cols].copy()

    # Short display names for width control
    rename = {
        "rank": "Rank",
        FUEL_COLUMN: "Fuel",
        "pred_Pressure": "Pred_P",
        "pred_rdot_mm_s": "Pred_rdot",
        "pred_Isp": "Pred_Isp",
        "pred_Thrust": "Pred_Thrust",
        "pred_cstar": "Pred_cstar",
        "uncertainty_rdot": "Unc_rdot",
        "final_score": "Score",
    }
    display_cols = [rename.get(c, c) for c in cols]

    # Fixed widths
    widths = {
        "Rank": 6,
        "Fuel": 12,
        "Pred_P": 10,
        "Pred_rdot": 10,
        "Pred_Isp": 10,
        "Pred_Thrust": 12,
        "Pred_cstar": 12,
        "Unc_rdot": 10,
        "Score": 10,
    }

    def fmt_cell(name, val):
        w = widths.get(name, 12)
        if name in ("Rank", "Fuel"):
            return f"{str(val):<{w}}"
        try:
            x = float(val)
            s = f"{x:.4f}" if np.isfinite(x) else "nan"
        except (TypeError, ValueError):
            s = str(val)
        return f"{s:>{w}}"

    header = "".join(
        f"{display_cols[i]:<{widths.get(display_cols[i], 12)}}" if display_cols[i] in ("Rank", "Fuel")
        else f"{display_cols[i]:>{widths.get(display_cols[i], 12)}}"
        for i in range(len(display_cols))
    )
    # rebuild header more carefully
    parts = []
    for dc in display_cols:
        w = widths.get(dc, 12)
        if dc in ("Rank", "Fuel"):
            parts.append(f"{dc:<{w}}")
        else:
            parts.append(f"{dc:>{w}}")
    header = "".join(parts)
    sep = "-" * len(header)

    print(f"\n========== {title} ==========")
    print(header)
    print(sep)
    for _, row in view.iterrows():
        line = ""
        for c, dc in zip(cols, display_cols):
            line += fmt_cell(dc, row[c])
        print(line)
    print(sep)

    # also return string-formatted frame for text export
    out_rows = []
    for _, row in view.iterrows():
        d = {}
        for c, dc in zip(cols, display_cols):
            if dc in ("Rank", "Fuel"):
                d[dc] = str(row[c])
            else:
                try:
                    x = float(row[c])
                    d[dc] = f"{x:.4f}" if np.isfinite(x) else "nan"
                except (TypeError, ValueError):
                    d[dc] = str(row[c])
        out_rows.append(d)
    return pd.DataFrame(out_rows)

def loo_by_firing(labeled, feature_cols):
    """
    Leave-One-Firing-Out:
    each unique fuel_type / sheet is held out once;
    train on all other firings; predict held-out rows.
    Also trains MC Dropout per fold for rdot uncertainty on held-out rows.
    """
    X = labeled[feature_cols + [FUEL_COLUMN]].reset_index(drop=True)
    y = labeled[TARGET_COLUMNS].to_numpy(dtype=float)
    groups = labeled[FUEL_COLUMN].to_numpy()
    rdot_idx = TARGET_COLUMNS.index(RDOT_TARGET)

    logo = LeaveOneGroupOut()
    n = len(X)
    y_hat = np.zeros_like(y, dtype=float)
    mc_std = np.zeros(n, dtype=float)
    fold_id = np.zeros(n, dtype=int)

    print(f"LOO folds (one firing held out per fold): {len(np.unique(groups))}")
    for fold, (tr, te) in enumerate(logo.split(X, y, groups), start=1):
        held = groups[te[0]]
        print(f"  Fold {fold}: hold out '{held}'  (train={len(tr)}, test={len(te)})")
        pipe = build_model_pipeline(SEED + fold, feature_cols)
        pipe.fit(X.iloc[tr], y[tr])
        y_hat[te] = pipe.predict(X.iloc[te])
        fold_id[te] = fold

        # MC Dropout on this fold (rdot only)
        x_tr = X.iloc[tr]
        x_te = X.iloc[te]
        train_nn, test_nn, _, _ = build_mc_dropout_inputs(x_tr, x_te, feature_cols)
        nn = train_mc_dropout_network(train_nn, y[tr, rdot_idx].astype(np.float32))
        _, std_te = mc_dropout_predict(nn, test_nn, N_MC_SAMPLES)
        mc_std[te] = std_te

    # Final model on ALL labeled data (for full-dataset predictions / export)
    final_pipe = build_model_pipeline(SEED, feature_cols)
    final_pipe.fit(X, y)

    return y_hat, mc_std, fold_id, final_pipe



def build_model_pipeline(seed, feature_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("fuel", make_one_hot_encoder(), [FUEL_COLUMN]),
            ("numeric", "passthrough", feature_cols),
        ],
        remainder="drop",
    )
    base = XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        min_child_weight=MIN_CHILD_WEIGHT,
        reg_alpha=REG_ALPHA,
        reg_lambda=REG_LAMBDA,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", MultiOutputRegressor(base))])


In [ ]:
# ============================================================
# 7. MAIN
# ============================================================

def main():
    if RDOT_TARGET not in TARGET_COLUMNS:
        raise ValueError(f"{RDOT_TARGET} must be in TARGET_COLUMNS")

    data, sheet_names, labeled_mask, gox_col = load_dataset()
    feature_cols = list(FEATURE_COLUMNS)
    labeled = data.loc[labeled_mask].reset_index(drop=True)

    y_true = labeled[TARGET_COLUMNS].to_numpy(dtype=float)
    y_hat, mc_std, fold_id, final_pipe = loo_by_firing(labeled, feature_cols)

    # LOO metrics on all labeled rows (each row predicted when its firing was held out)
    test_r2 = safe_r2_multi(y_true, y_hat)
    test_mae = mean_absolute_error(y_true, y_hat, multioutput="raw_values")
    test_rmse = np.sqrt(mean_squared_error(y_true, y_hat, multioutput="raw_values"))

    metrics_df = print_metrics_table(test_r2, test_mae, test_rmse, TARGET_COLUMNS, title=f"LOO METRICS ({MODEL_NAME})")

    rdot_idx = TARGET_COLUMNS.index(RDOT_TARGET)
    mc_r2 = (r2_score(y_true[:, rdot_idx], y_hat[:, rdot_idx]) if np.unique(y_true[:, rdot_idx]).size > 1 else np.nan)
    # MC mean is not stored separately; tree LOO preds used for rdot point estimate
    mc_mae = mean_absolute_error(y_true[:, rdot_idx], y_hat[:, rdot_idx])
    print(
        f"MC Dropout LOO mean sigma={mc_std.mean():.4f} | "
        f"rdot LOO MAE={mc_mae:.4f}"
    )

    # Attach LOO predictions to labeled rows; full data gets final-model predictions
    loo_results = labeled.copy()
    for i, t in enumerate(TARGET_COLUMNS):
        loo_results[f"pred_{t}"] = y_hat[:, i]
    loo_results["uncertainty_rdot"] = mc_std
    loo_results["loo_fold"] = fold_id

    x_all = data[feature_cols + [FUEL_COLUMN]].copy()
    all_pred = final_pipe.predict(x_all)
    results = data.copy()
    for i, t in enumerate(TARGET_COLUMNS):
        results[f"pred_{t}"] = all_pred[:, i]
    # map LOO uncertainty by index where possible
    results["uncertainty_rdot"] = np.nan
    results.loc[labeled_mask.values, "uncertainty_rdot"] = mc_std

    # Rank firings using LOO predictions (honest out-of-firing estimates)
    pred_columns = [f"pred_{t}" for t in TARGET_COLUMNS]
    agg = {c: "mean" for c in pred_columns}
    agg["uncertainty_rdot"] = "mean"
    summary = loo_results.groupby(FUEL_COLUMN, as_index=False).agg(agg)

    for t in TARGET_COLUMNS:
        summary[f"norm_{t}"] = minmax_norm(summary[f"pred_{t}"])
    wsum = sum(TARGET_WEIGHTS.values())
    summary["performance_score"] = (sum(TARGET_WEIGHTS[t] * summary[f"norm_{t}"] for t in TARGET_COLUMNS) / wsum)
    summary["uncertainty_norm"] = minmax_norm(summary["uncertainty_rdot"])
    summary["final_score"] = (
        summary["performance_score"] - UNCERTAINTY_PENALTY_K * summary["uncertainty_norm"]
    )
    summary = summary.sort_values("final_score", ascending=False).reset_index(drop=True)
    summary["rank"] = np.arange(1, len(summary) + 1)
    best = summary.iloc[0]

    show = ["rank", FUEL_COLUMN] + pred_columns + ["uncertainty_rdot", "final_score"]
    ranking_view = print_ranking_table(
        summary, pred_columns, title=f"FUEL RANKING (LOO) — {MODEL_NAME}"
    )

    print(f"\n========== BEST FUEL ({MODEL_NAME}, LOO) ==========")
    print("Best Fuel               :", best[FUEL_COLUMN])
    print(
        "Regression Rate         : {:.4f} ± {:.4f} mm/s".format(
            float(best[PRED_RDOT]), float(best["uncertainty_rdot"])
        )
    )
    print("Pressure (pred)         : {:.4f}".format(float(best["pred_Pressure"])))
    print("Specific Impulse        : {:.2f}".format(float(best["pred_Isp"])))
    print("Thrust                  : {:.2f}".format(float(best["pred_Thrust"])))
    print("Characteristic Velocity : {:.2f}".format(float(best["pred_cstar"])))
    print("Final Score             : {:.4f}".format(float(best["final_score"])))

    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        results.to_excel(writer, sheet_name="All_Predictions", index=False)
        loo_results.to_excel(writer, sheet_name="LOO_Predictions", index=False)
        summary.to_excel(writer, sheet_name="Fuel_Ranking_LOO", index=False)

    lines = [
        f"HYBRID ROCKET FUEL BLEND — LOO SUMMARY ({MODEL_NAME})",
        "=" * 60,
        f"Main model     : {MODEL_NAME}",
        f"Validation     : Leave-One-Firing-Out (by {FUEL_COLUMN})",
        f"Excel file     : {EXCEL_PATH}",
        f"Sheets used    : {sheet_names}",
        f"Input features : {feature_cols}",
        f"Targets        : {TARGET_COLUMNS}",
        f"Labeled rows   : {len(labeled)}",
        "",
        "LOO PER-TARGET METRICS",
        "-" * 60,
        metrics_df.to_string(index=False),
    ]
    lines += [
        "",
        f"MC Dropout mean sigma (LOO folds) = {mc_std.mean():.4f}",
        "",
        "FUEL RANKING (LOO)",
        "-" * 60,
        ranking_view.to_string(index=False),
        "",
        "BEST FUEL",
        "-" * 60,
        f"Best Fuel               : {best[FUEL_COLUMN]}",
        f"Regression Rate         : {float(best[PRED_RDOT]):.4f} ± "
        f"{float(best['uncertainty_rdot']):.4f} mm/s",
        f"Pressure (pred)         : {float(best['pred_Pressure']):.4f}",
        f"Specific Impulse        : {float(best['pred_Isp']):.2f}",
        f"Thrust                  : {float(best['pred_Thrust']):.2f}",
        f"Characteristic Velocity : {float(best['pred_cstar']):.2f}",
        f"Final Score             : {float(best['final_score']):.4f}",
        "",
        f"Excel : {OUTPUT_EXCEL}",
    ]
    OUTPUT_TXT.write_text("\n".join(lines), encoding="utf-8")
    print(f"\nSummary: {OUTPUT_TXT.resolve()}")
    print(f"Excel  : {OUTPUT_EXCEL.resolve()}")


if __name__ == "__main__":
    main()
